# H1 Gates on UPMC Food-101 (RebQ/PAL Protocol: 100 Classes, 10 Tasks)
- **Dataset:** **UPMC Food-101** (100 classes after dropping rarest class per RebQ/PAL protocol, 10 class-incremental tasks × 10 classes/task).
- **Hardware:** Optimized for **Tesla T4 GPU** (`FP16 + GradScaler(init_scale=1024.0)`, `BS=64`, `np.memmap` disk-backed image tensor so System RAM stays `< 2.5 GB`).
- **Outputs:** Written to `/content/h1_results/food101` (`food_results.json`, `food_predictions.npz`, `run.log`), zipped and downloaded automatically after `G1` and after `G2`.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, sys; print("CPUs:", os.cpu_count())


In [ ]:
import pathlib
FILES = {"mm_flex_merge/__init__.py": "", "mm_flex_merge/models/__init__.py": "from .analytic import BlockAnalyticCL\n", "mm_flex_merge/models/analytic.py": "\"\"\"Block-structured analytic continual learner on frozen or Null-Space LoRA multimodal embeddings.\n\nEach modality m has its own frozen random projection, phi_m(x) = relu(normalize(x) @ W_m).\nFor M=2 core modalities (Image 'img' and Text 'txt'), the model maintains sufficient statistics\nover the block-concatenated features phi = [phi_img, phi_txt]:\n\n    G = [[G_II, G_IT], [G_TI, G_TT]],   S = [[S_I], [S_T]]   (+ per-class Grams/counts when beta > 0)\n\nConsequences (all exact in FP64, zero retraining):\n  * update / delete        : add or subtract statistics -> deletion == retraining without the data.\n  * order invariance       : statistics are commutative sums, so task order does not matter.\n  * missing modality       : solve ridge on the Gram sub-block of the available modalities,\n                             which equals a dedicated model trained on those modalities only.\n  * asymmetric ridge       : modality-specific Tikhonov regularization (gamma_I, gamma_T) scaled\n                             by each block's own trace Tr(G_mm) / P.\n  * Block Schur Complement : solves the 2P x 2P system via P x P inversions while cancelling\n                             cross-modal collinearity (-rho * G_IT * W_T) and reusing W_T.\n  * CMML / MIL availability: accumulates partial-modality observations and normalizes block energy\n                             by per-modality observation counts N_{c, m}.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport zlib\nfrom typing import Mapping, Sequence\n\nimport torch\nimport torch.nn.functional as F\n\n\nclass BlockAnalyticCL:\n    def __init__(\n        self,\n        modality_dims: Mapping[str, int],\n        num_classes: int,\n        *,\n        projection_dim: int = 1024,\n        ridge: float | Mapping[str, float] = 0.1,\n        class_balance_beta: float = 0.0,\n        schur_coupling_rho: float = 1.0,\n        seed: int = 2026,\n    ) -> None:\n        if not modality_dims or num_classes <= 0 or projection_dim <= 0:\n            raise ValueError(\"invalid BlockAnalyticCL configuration\")\n        if isinstance(ridge, Mapping):\n            if any(float(v) <= 0 for v in ridge.values()):\n                raise ValueError(\"ridge values must be positive\")\n            self.modality_ridges: dict[str, float] | None = {str(k): float(v) for k, v in ridge.items()}\n            self.ridge = float(next(iter(self.modality_ridges.values())))\n        else:\n            if float(ridge) <= 0:\n                raise ValueError(\"ridge must be positive\")\n            self.modality_ridges = None\n            self.ridge = float(ridge)\n        if class_balance_beta < 0:\n            raise ValueError(\"class_balance_beta must be non-negative\")\n        if not (0.0 <= float(schur_coupling_rho) <= 1.0):\n            raise ValueError(\"schur_coupling_rho must be in [0, 1]\")\n\n        self.modalities = tuple(modality_dims)\n        self.num_classes = int(num_classes)\n        self.projection_dim = int(projection_dim)\n        self.beta = float(class_balance_beta)\n        self.rho = float(schur_coupling_rho)\n\n        # Per-modality seed depends only on (seed, name) so a single-modality model\n        # shares the exact projection of the same block in a multimodal model.\n        self.projections: dict[str, torch.Tensor] = {}\n        for name, dim in modality_dims.items():\n            g = torch.Generator().manual_seed(int(seed) + zlib.crc32(name.encode()))\n            self.projections[name] = torch.randn(int(dim), self.projection_dim, generator=g, dtype=torch.float64)\n\n        D = self.projection_dim * len(self.modalities)\n        self.gram = torch.zeros(D, D, dtype=torch.float64)\n        self.class_sums = torch.zeros(D, self.num_classes, dtype=torch.float64)\n        self.class_grams = torch.zeros(self.num_classes, D, D, dtype=torch.float64) if self.beta > 0 else None\n        self.class_counts = torch.zeros(self.num_classes, dtype=torch.long)\n        self.modality_class_counts = torch.zeros(len(self.modalities), self.num_classes, dtype=torch.long)\n        self._has_partial_observations = False\n        self._cache: dict[tuple[str, ...], torch.Tensor] = {}\n\n    # ------------------------------------------------------------------ features\n    def _check(self, modalities: Sequence[str] | None) -> tuple[str, ...]:\n        if modalities is None:\n            return self.modalities\n        mods = tuple(m for m in self.modalities if m in set(modalities))\n        if not mods or len(mods) != len(set(modalities)):\n            raise ValueError(f\"unknown or empty modalities {modalities}; known {self.modalities}\")\n        return mods\n\n    def _indices(self, mods: tuple[str, ...]) -> torch.Tensor:\n        P = self.projection_dim\n        return torch.cat([torch.arange(self.modalities.index(m) * P, (self.modalities.index(m) + 1) * P) for m in mods])\n\n    def features(self, inputs: Mapping[str, torch.Tensor], modalities: Sequence[str] | None = None) -> torch.Tensor:\n        mods = self._check(modalities)\n        return torch.cat(\n            [F.relu(F.normalize(inputs[m].double(), dim=-1) @ self.projections[m]) for m in mods], dim=1\n        )\n\n    # ---------------------------------------------------------------- statistics\n    def _accumulate(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        labels: torch.Tensor,\n        sign: float,\n        availability: Mapping[str, torch.Tensor] | None = None,\n    ) -> None:\n        labels = labels.long()\n        n = int(labels.shape[0])\n        P = self.projection_dim\n\n        # Support both full-modality inputs and partial-modality (CMML / MIL) inputs\n        present_mods = [m for m in self.modalities if m in inputs and inputs[m] is not None]\n        if not present_mods:\n            raise ValueError(\"at least one known modality must be provided in inputs\")\n\n        phi_blocks = []\n        avail_masks = []\n        for m_idx, m in enumerate(self.modalities):\n            if m in inputs and inputs[m] is not None:\n                block = F.relu(F.normalize(inputs[m].double(), dim=-1) @ self.projections[m])\n                if availability is not None and m in availability and availability[m] is not None:\n                    mask = availability[m].bool().view(-1)\n                    if not bool(mask.all()):\n                        self._has_partial_observations = True\n                        block = torch.where(mask.unsqueeze(1), block, torch.zeros_like(block))\n                else:\n                    mask = torch.ones(n, dtype=torch.bool)\n            else:\n                self._has_partial_observations = True\n                block = torch.zeros(n, P, dtype=torch.float64)\n                mask = torch.zeros(n, dtype=torch.bool)\n            phi_blocks.append(block)\n            avail_masks.append(mask)\n\n        phi = torch.cat(phi_blocks, dim=1)\n        counts = torch.bincount(labels, minlength=self.num_classes)\n        if sign < 0 and bool((counts > self.class_counts).any()):\n            raise ValueError(\"cannot delete more samples of a class than were observed\")\n\n        self.gram.add_(phi.T @ phi, alpha=sign)\n        self.class_sums.add_(phi.T @ F.one_hot(labels, self.num_classes).double(), alpha=sign)\n        self.class_counts.add_(counts if sign > 0 else -counts)\n        for m_idx, mask in enumerate(avail_masks):\n            if bool(mask.any()):\n                m_counts = torch.bincount(labels[mask], minlength=self.num_classes)\n                self.modality_class_counts[m_idx].add_(m_counts if sign > 0 else -m_counts)\n\n        if self.class_grams is not None:\n            for c in labels.unique().tolist():\n                f = phi[labels == c]\n                self.class_grams[c].add_(f.T @ f, alpha=sign)\n        self._cache.clear()\n\n    def update(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        labels: torch.Tensor,\n        availability: Mapping[str, torch.Tensor] | None = None,\n    ) -> None:\n        self._accumulate(inputs, labels, 1.0, availability=availability)\n\n    def delete(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        labels: torch.Tensor,\n        availability: Mapping[str, torch.Tensor] | None = None,\n    ) -> None:\n        \"\"\"Exact unlearning: remove previously seen samples (any granularity: sample/task).\"\"\"\n        self._accumulate(inputs, labels, -1.0, availability=availability)\n\n    def class_weights(self, counts: torch.Tensor | None = None) -> torch.Tensor:\n        cnt = (self.class_counts if counts is None else counts).double()\n        weights = torch.zeros_like(cnt)\n        observed = cnt > 0\n        if observed.any():\n            rel = cnt[observed] / cnt[observed].mean()\n            weights[observed] = rel.pow(-self.beta)\n            weights[observed] /= weights[observed].mean()\n        return weights\n\n    # --------------------------------------------------------------------- solve\n    def _weighted_gram_and_sums(self, mods: tuple[str, ...]) -> tuple[torch.Tensor, torch.Tensor]:\n        idx = self._indices(mods)\n        P = self.projection_dim\n        if self.class_grams is None:\n            return self.gram[idx][:, idx].clone(), self.class_sums[idx].clone()\n\n        w = self.class_weights()\n        if not self._has_partial_observations or len(mods) == 1:\n            if len(mods) == 1 and self._has_partial_observations:\n                m_idx = self.modalities.index(mods[0])\n                w = self.class_weights(self.modality_class_counts[m_idx])\n            G = torch.einsum(\"c,cij->ij\", w, self.class_grams[:, idx][:, :, idx])\n            S = self.class_sums[idx] * w.unsqueeze(0)\n            return G, S\n\n        # Availability-normalized weighting across modalities when partial observations exist\n        scales = []\n        for m in mods:\n            m_idx = self.modalities.index(m)\n            mc = self.modality_class_counts[m_idx].double()\n            sc = torch.where(mc > 0, self.class_counts.double() / mc.clamp(min=1.0), torch.ones_like(w))\n            scales.append(sc)\n\n        K = len(mods)\n        G = torch.zeros(K * P, K * P, dtype=torch.float64)\n        S = torch.zeros(K * P, self.num_classes, dtype=torch.float64)\n        sub_cgrams = self.class_grams[:, idx][:, :, idx]\n        sub_csums = self.class_sums[idx]\n        for a in range(K):\n            sa = slice(a * P, (a + 1) * P)\n            wa = w * scales[a]\n            G[sa, sa] = torch.einsum(\"c,cij->ij\", wa, sub_cgrams[:, sa, sa])\n            S[sa] = sub_csums[sa] * wa.unsqueeze(0)\n            for b in range(a + 1, K):\n                sb = slice(b * P, (b + 1) * P)\n                wab = w * torch.sqrt(scales[a] * scales[b])\n                G[sa, sb] = torch.einsum(\"c,cij->ij\", wab, sub_cgrams[:, sa, sb])\n                G[sb, sa] = G[sa, sb].T\n        return G, S\n\n    def solve_schur(\n        self,\n        gamma_first: float | None = None,\n        gamma_second: float | None = None,\n        rho: float | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        \"\"\"\n        Optimal 2x2 Block Schur Complement solver for M=2 modalities (e.g. Image and Text).\n        Returns (W_both, W_first_only, W_second_only) in O(P^3) by reusing the unimodal inverse:\n          A_2 = G_22 + lam_2 I_P\n          W_2_only = A_2^{-1} S_2\n          Schur_1 = (G_11 + lam_1 I_P) - rho^2 * G_12 A_2^{-1} G_21\n          W_1_joint = Schur_1^{-1} (S_1 - rho * G_12 W_2_only)\n          W_2_joint = W_2_only - rho * A_2^{-1} G_21 W_1_joint\n        \"\"\"\n        if len(self.modalities) != 2:\n            raise ValueError(\"solve_schur requires exactly 2 modalities\")\n        if int(self.class_counts.sum()) <= 0:\n            raise RuntimeError(\"BlockAnalyticCL needs at least one observed sample\")\n\n        P = self.projection_dim\n        m1, m2 = self.modalities\n        g1 = float(gamma_first if gamma_first is not None else (self.modality_ridges.get(m1, self.ridge) if self.modality_ridges else self.ridge))\n        g2 = float(gamma_second if gamma_second is not None else (self.modality_ridges.get(m2, self.ridge) if self.modality_ridges else self.ridge))\n        rho_val = float(self.rho if rho is None else rho)\n\n        G, S = self._weighted_gram_and_sums(self.modalities)\n        G_11, G_22, G_12 = G[:P, :P], G[P:, P:], G[:P, P:]\n        S_1, S_2 = S[:P], S[P:]\n\n        tr_1_uni = max(float(torch.trace(G_11)) / P, 1e-12)\n        tr_2_uni = max(float(torch.trace(G_22)) / P, 1e-12)\n        use_block_trace = (self.modality_ridges is not None) or (gamma_first is not None) or (gamma_second is not None)\n        if use_block_trace:\n            tr_1, tr_2 = tr_1_uni, tr_2_uni\n        else:\n            tr_joint = max(float(torch.trace(G)) / (2 * P), 1e-12)\n            tr_1 = tr_2 = tr_joint\n\n        eye = torch.eye(P, dtype=G.dtype)\n        A_1 = G_11 + (g1 * tr_1) * eye\n        A_2 = G_22 + (g2 * tr_2) * eye\n\n        sol_2 = torch.linalg.solve(A_2, torch.cat([G_12.T, S_2], dim=1))\n        A_2_inv_G_21 = sol_2[:, :P]\n        W_2_joint_base = sol_2[:, P:]\n\n        if use_block_trace:\n            W_1_only = torch.linalg.solve(A_1, S_1)\n            W_2_only = W_2_joint_base\n        else:\n            W_1_only = torch.linalg.solve(G_11 + (g1 * tr_1_uni) * eye, S_1)\n            W_2_only = torch.linalg.solve(G_22 + (g2 * tr_2_uni) * eye, S_2)\n\n        if rho_val == 0.0:\n            W_1_joint = W_1_only if use_block_trace else torch.linalg.solve(A_1, S_1)\n            W_both = torch.cat([W_1_joint, W_2_joint_base], dim=0)\n        else:\n            Schur_1 = A_1 - (rho_val ** 2) * (G_12 @ A_2_inv_G_21)\n            rhs_1 = S_1 - rho_val * (G_12 @ W_2_joint_base)\n            W_1_joint = torch.linalg.solve(Schur_1, rhs_1)\n            W_2_joint = W_2_joint_base - rho_val * (A_2_inv_G_21 @ W_1_joint)\n            W_both = torch.cat([W_1_joint, W_2_joint], dim=0)\n\n        return W_both, W_1_only, W_2_only\n\n    def auto_spectral_ridges(self, gamma_base: float = 0.35) -> dict[str, float]:\n        \"\"\"\n        Pillar 2: Self-tuning Empirical-Bayes Noise-to-Signal & Sample-Load Shrinkage Law.\n        Automatically computes modality-specific regularization (gamma_1, gamma_2) from\n        the accumulated class-balanced Gram blocks (G_11, G_22, S_1, S_2) and sample counts.\n        \"\"\"\n        if len(self.modalities) != 2:\n            raise ValueError(\"auto_spectral_ridges requires exactly 2 modalities\")\n        if int(self.class_counts.sum()) <= 0:\n            raise RuntimeError(\"BlockAnalyticCL needs at least one observed sample\")\n\n        P = self.projection_dim\n        m1, m2 = self.modalities\n        G, S = self._weighted_gram_and_sums(self.modalities)\n        G_11, G_22 = G[:P, :P], G[P:, P:]\n        S_1, S_2 = S[:P], S[P:]\n\n        tr_1 = max(float(torch.trace(G_11)), 1e-12)\n        tr_2 = max(float(torch.trace(G_22)), 1e-12)\n        sig_1 = max(float(torch.sum(S_1 * S_1)), 1e-12)\n        sig_2 = max(float(torch.sum(S_2 * S_2)), 1e-12)\n\n        nsr_1 = (tr_1 * tr_1) / (P * sig_1)\n        nsr_2 = (tr_2 * tr_2) / (P * sig_2)\n        nsr_geom = (nsr_1 * nsr_2) ** 0.5\n\n        n_1 = max(int(self.modality_class_counts[0].sum().item()), 1)\n        n_2 = max(int(self.modality_class_counts[1].sum().item()), 1)\n        load_1 = (float(2 * P) / n_1) ** 1.25\n        load_2 = (float(2 * P) / n_2) ** 1.25\n\n        eb_1 = (nsr_1 / max(nsr_geom, 1e-12)) ** 2.0\n        eb_2 = (nsr_2 / max(nsr_geom, 1e-12)) ** 2.0\n\n        g1 = float(min(max(gamma_base * load_1 * eb_1, 0.02), 5.0))\n        g2 = float(min(max(gamma_base * load_2 * eb_2, 0.02), 5.0))\n        self.modality_ridges = {m1: g1, m2: g2}\n        self._cache.clear()\n        return dict(self.modality_ridges)\n\n    def schur_innovation_decomposition(\n        self,\n        gamma_first: float | None = None,\n        gamma_second: float | None = None,\n        rho: float | None = None,\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:\n        \"\"\"\n        Pillar 1: Exact Schur-Innovation Decomposition (SID).\n        Decomposes the joint 2x2 Block Schur estimator into its marginal unimodal estimators\n        plus the exact cross-modal Schur innovation matrices:\n          W_1_joint = W_1_only + dW_1_schur\n          W_2_joint = W_2_only + dW_2_schur\n        Returns (W_1_only, W_2_only, dW_1_schur, dW_2_schur, W_both).\n        \"\"\"\n        if len(self.modalities) != 2:\n            raise ValueError(\"schur_innovation_decomposition requires exactly 2 modalities\")\n        if int(self.class_counts.sum()) <= 0:\n            raise RuntimeError(\"BlockAnalyticCL needs at least one observed sample\")\n\n        P = self.projection_dim\n        m1, m2 = self.modalities\n        g1 = float(gamma_first if gamma_first is not None else (self.modality_ridges.get(m1, self.ridge) if self.modality_ridges else self.ridge))\n        g2 = float(gamma_second if gamma_second is not None else (self.modality_ridges.get(m2, self.ridge) if self.modality_ridges else self.ridge))\n        rho_val = float(self.rho if rho is None else rho)\n\n        G, S = self._weighted_gram_and_sums(self.modalities)\n        G_11, G_22, G_12 = G[:P, :P], G[P:, P:], G[:P, P:]\n        S_1, S_2 = S[:P], S[P:]\n\n        use_block_trace = (self.modality_ridges is not None) or (gamma_first is not None) or (gamma_second is not None)\n        if use_block_trace:\n            tr_1 = max(float(torch.trace(G_11)) / P, 1e-12)\n            tr_2 = max(float(torch.trace(G_22)) / P, 1e-12)\n        else:\n            tr_joint = max(float(torch.trace(G)) / (2 * P), 1e-12)\n            tr_1 = tr_2 = tr_joint\n\n        eye = torch.eye(P, dtype=G.dtype)\n        A_1 = G_11 + (g1 * tr_1) * eye\n        A_2 = G_22 + (g2 * tr_2) * eye\n\n        W_1_only = torch.linalg.solve(A_1, S_1)\n        sol_2 = torch.linalg.solve(A_2, torch.cat([G_12.T, S_2], dim=1))\n        A_2_inv_G_21 = sol_2[:, :P]\n        W_2_only = sol_2[:, P:]\n\n        if rho_val == 0.0:\n            dW_1_schur = torch.zeros_like(W_1_only)\n            dW_2_schur = torch.zeros_like(W_2_only)\n        else:\n            Schur_1 = A_1 - (rho_val ** 2) * (G_12 @ A_2_inv_G_21)\n            innov_rhs_1 = (rho_val ** 2) * (G_12 @ (A_2_inv_G_21 @ W_1_only)) - rho_val * (G_12 @ W_2_only)\n            dW_1_schur = torch.linalg.solve(Schur_1, innov_rhs_1)\n            dW_2_schur = -rho_val * (A_2_inv_G_21 @ (W_1_only + dW_1_schur))\n\n        W_both = torch.cat([W_1_only + dW_1_schur, W_2_only + dW_2_schur], dim=0)\n        return W_1_only, W_2_only, dW_1_schur, dW_2_schur, W_both\n\n    def transport_gram(\n        self,\n        old_inputs: Mapping[str, torch.Tensor],\n        new_inputs: Mapping[str, torch.Tensor],\n        ridge_transport: float = 1e-4,\n        max_drift: float = 0.35,\n        adaptive_drift_guard: bool = True,\n    ) -> dict[str, torch.Tensor]:\n        \"\"\"\n        Pillar 3: Exemplar-Free Block-Gram Pushforward Transport with Perturbative Drift Guard.\n        Estimates modality-specific linear transport operators M_m = I + eta_m * dM_m from\n        current-task feature shifts (old_inputs -> new_inputs) and transforms all historical\n        class Gram blocks and class sums in-place: G_c <- M G_c M^T, S_c <- M S_c (B = 0).\n        When `adaptive_drift_guard=True`, automatically damps or bypasses transport if the\n        relative feature drift exceeds the first-order perturbative trust region `max_drift`.\n        \"\"\"\n        P = self.projection_dim\n        eye = torch.eye(P, dtype=torch.float64)\n        operators: dict[str, torch.Tensor] = {}\n\n        for m in self.modalities:\n            if m in old_inputs and m in new_inputs and old_inputs[m] is not None and new_inputs[m] is not None:\n                z_old = self.features(old_inputs, (m,))\n                z_new = self.features(new_inputs, (m,))\n                diff = z_new - z_old\n                delta_rel = float(torch.linalg.norm(diff) / torch.linalg.norm(z_old).clamp(min=1e-12))\n                if adaptive_drift_guard and delta_rel >= max_drift:\n                    operators[m] = eye\n                    continue\n                cov_old = z_old.T @ z_old\n                tr_scale = max(float(torch.trace(cov_old)) / P, 1e-12)\n                if adaptive_drift_guard and max_drift > 0:\n                    ratio2 = (delta_rel / max_drift) ** 2\n                    eff_ridge = ridge_transport + 0.05 * ratio2\n                    eta = 1.0 - ratio2\n                else:\n                    eff_ridge = ridge_transport\n                    eta = 1.0\n                lam = eff_ridge * tr_scale\n                dM_T = torch.linalg.solve(cov_old + lam * eye, z_old.T @ diff)\n                operators[m] = eye + eta * dM_T.T\n            else:\n                operators[m] = eye\n\n        M_blocks = [operators[m] for m in self.modalities]\n        M_full = torch.block_diag(*M_blocks)\n        if self.class_grams is not None:\n            for c in range(self.num_classes):\n                if int(self.class_counts[c].item()) > 0:\n                    self.class_grams[c] = M_full @ self.class_grams[c] @ M_full.T\n                    self.class_sums[:, c] = M_full @ self.class_sums[:, c]\n            self.gram = self.class_grams.sum(dim=0)\n        else:\n            self.gram = M_full @ self.gram @ M_full.T\n            self.class_sums = M_full @ self.class_sums\n        self._cache.clear()\n        return operators\n\n    @staticmethod\n    def auto_continual_policy(\n        g1_gain_pts: float | None = None,\n        incremental_mode: str = \"dil\",\n        min_g1_gain_pts: float = 2.0,\n    ) -> dict[str, object]:\n        \"\"\"\n        Self-governing continual adaptation policy derived from cross-dataset empirical laws\n        (CrisisMMD v2.0 DIL vs. UPMC Food-101 100-class CIL):\n          1. If `g1_gain_pts < min_g1_gain_pts` (backbone already aligned with target domain):\n             keep backbone frozen (`tau=1.0`) and use `BlockAnalyticCL` with `adaptive_sid=True`.\n          2. If `incremental_mode == 'dil'` (Domain-Incremental, fixed class set):\n             adapt Vision tower in null space (`tau=0.95`) while keeping Text tower as frozen\n             semantic anchor (`towers=('vision',)`).\n          3. If `incremental_mode == 'cil'` (Class-Incremental, expanding class set):\n             adapt both towers with strict null-space protection (`tau=0.99`) + Gram transport & SID.\n        \"\"\"\n        mode = incremental_mode.strip().lower()\n        if mode not in {\"dil\", \"cil\"}:\n            raise ValueError(f\"incremental_mode must be 'dil' or 'cil', got {incremental_mode!r}\")\n        if g1_gain_pts is not None and float(g1_gain_pts) < min_g1_gain_pts:\n            return {\n                \"regime\": \"frozen_analytic_sid\",\n                \"lora_enabled\": False,\n                \"tau\": 1.0,\n                \"towers\": (),\n                \"use_transport\": False,\n                \"adaptive_sid\": True,\n            }\n        if mode == \"dil\":\n            return {\n                \"regime\": \"dil_vision_nullspace_frozen_text\",\n                \"lora_enabled\": True,\n                \"tau\": 0.95,\n                \"towers\": (\"vision\",),\n                \"use_transport\": True,\n                \"adaptive_sid\": True,\n            }\n        return {\n            \"regime\": \"cil_dual_nullspace_transport_sid\",\n            \"lora_enabled\": True,\n            \"tau\": 0.99,\n            \"towers\": (\"vision\", \"text\"),\n            \"use_transport\": True,\n            \"adaptive_sid\": True,\n        }\n\n    def solve(self, modalities: Sequence[str] | None = None) -> torch.Tensor:\n        mods = self._check(modalities)\n        if mods not in self._cache:\n            if int(self.class_counts.sum()) <= 0:\n                raise RuntimeError(\"BlockAnalyticCL needs at least one observed sample\")\n            if len(mods) == 2 and len(self.modalities) == 2 and (self.modality_ridges is not None or self.rho != 1.0):\n                W_both, W_1, W_2 = self.solve_schur()\n                self._cache[self.modalities] = W_both\n                self._cache[(self.modalities[0],)] = W_1\n                self._cache[(self.modalities[1],)] = W_2\n                return self._cache[mods]\n\n            G, S = self._weighted_gram_and_sums(mods)\n            P = self.projection_dim\n            if self.modality_ridges is not None and len(mods) == 1:\n                r_m = self.modality_ridges.get(mods[0], self.ridge)\n                lam = r_m * max(float(torch.trace(G)) / G.shape[0], 1e-12)\n            else:\n                lam = self.ridge * max(float(torch.trace(G)) / G.shape[0], 1e-12)\n            self._cache[mods] = torch.linalg.solve(G + lam * torch.eye(G.shape[0], dtype=G.dtype), S)\n        return self._cache[mods]\n\n    def logits(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        modalities: Sequence[str] | None = None,\n        adaptive_sid: bool = False,\n        tau_gate: float = 2.0,\n    ) -> torch.Tensor:\n        \"\"\"\n        Logits using only `modalities` (default: all present in `inputs`).\n        When `adaptive_sid=True` and both modalities are present, applies the continuous\n        Schur-Innovation Decomposition (SID) reliability-modulated inference law.\n        \"\"\"\n        if modalities is None:\n            present = tuple(m for m in self.modalities if m in inputs and inputs[m] is not None)\n            mods = self._check(present if present else None)\n        else:\n            mods = self._check(modalities)\n\n        if adaptive_sid and len(mods) == 2 and len(self.modalities) == 2:\n            m1, m2 = self.modalities\n            z_1 = self.features(inputs, (m1,))\n            z_2 = self.features(inputs, (m2,))\n            W_1_o, W_2_o, dW_1_s, dW_2_s, _ = self.schur_innovation_decomposition()\n\n            L_1_o = z_1 @ W_1_o\n            L_2_o = z_2 @ W_2_o\n            L_1_in = z_1 @ dW_1_s\n            L_2_in = z_2 @ dW_2_s\n\n            top2_1 = torch.topk(L_1_o, k=min(2, self.num_classes), dim=1).values\n            margin_1 = (top2_1[:, 0] - top2_1[:, -1]).clamp(min=1e-6)\n            top2_2 = torch.topk(L_2_o, k=min(2, self.num_classes), dim=1).values\n            margin_2 = (top2_2[:, 0] - top2_2[:, -1]).clamp(min=1e-6)\n\n            rel_1 = margin_1 / (0.5 * (margin_1 + margin_2))\n            rel_2 = margin_2 / (0.5 * (margin_1 + margin_2))\n            alpha_1 = torch.pow(rel_1, 1.0 / tau_gate).unsqueeze(1)\n            alpha_2 = torch.pow(rel_2, 1.0 / tau_gate).unsqueeze(1)\n\n            prob_1 = F.softmax(L_1_o * 4.0, dim=1)\n            prob_2 = F.softmax(L_2_o * 4.0, dim=1)\n            cos_agree = F.cosine_similarity(prob_1, prob_2, dim=1).clamp(min=0.0, max=1.0)\n            bal_ratio = (2.0 * torch.sqrt(margin_1 * margin_2) / (margin_1 + margin_2)).clamp(0.0, 1.0)\n            rho_x = (bal_ratio * (0.5 + 0.5 * cos_agree)).unsqueeze(1)\n\n            return alpha_1 * L_1_o + alpha_2 * L_2_o + rho_x * (L_1_in + L_2_in)\n\n        return self.features(inputs, mods) @ self.solve(mods)\n\n    def predict(\n        self,\n        inputs: Mapping[str, torch.Tensor],\n        modalities: Sequence[str] | None = None,\n        adaptive_sid: bool = False,\n        tau_gate: float = 2.0,\n    ) -> torch.Tensor:\n        return self.logits(inputs, modalities, adaptive_sid=adaptive_sid, tau_gate=tau_gate).argmax(dim=-1)\n", "h1_gate_engine.py": "\"\"\"\nSelf-contained engine for gates H1-G1 / H1-G2 (used locally and embedded verbatim in the Colab notebook).\n\nG1: adaptation gain of LoRA (both CLIP towers) vs frozen, analytic head (BlockAnalyticCL).\nG2: sequential LoRA over tasks; each session's LoRA input side is projected onto the null space of\n    the accumulated input Gram of previous sessions, analytic statistics are never recomputed.\nAll evaluation is on the validation split; the caller must drop the test split beforehand.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport time\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom PIL import Image\nfrom sklearn.metrics import accuracy_score, f1_score\n\nfrom mm_flex_merge.models import BlockAnalyticCL\n\nNAME = \"openai/clip-vit-base-patch16\"\nMEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1)\nSTD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1)\nR, ALPHA = 16, 32\nTEXT_MASK = {\"mask\": None}\nDEV = \"cuda\" if torch.cuda.is_available() else \"cpu\"\nif DEV == \"cuda\":\n    AMP = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16\nelse:\n    AMP = None\n\n\nLOG_PATH = None    # set by the notebook to a file inside the exported results folder\n\n\ndef log(msg):\n    line = f\"{time.strftime('%H:%M:%S')} {msg}\"\n    print(line, flush=True)\n    if LOG_PATH is not None:\n        with open(LOG_PATH, \"a\", encoding=\"utf-8\") as f:\n            f.write(line + \"\\n\")\n\n\n# ----------------------------------------------------------------------------- data\nclass _ImageSet(torch.utils.data.Dataset):\n    def __init__(self, paths):\n        self.paths = list(paths)\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, i):\n        try:\n            im = Image.open(self.paths[i]).convert(\"RGB\")\n        except Exception:\n            im = Image.new(\"RGB\", (224, 224))\n        w, h = im.size\n        s = 224 / min(w, h)\n        im = im.resize((max(224, round(w * s)), max(224, round(h * s))), Image.BICUBIC)\n        w, h = im.size\n        l, t = (w - 224) // 2, (h - 224) // 2\n        im = im.crop((l, t, l + 224, t + 224))\n        return torch.from_numpy(np.asarray(im, dtype=np.uint8).copy()).permute(2, 0, 1)\n\n\ndef decode_images(paths, workers=2, bs=256, mmap_path=None):\n    import tempfile\n    from pathlib import Path\n    w_eff = min(max(int(workers or 2), 0), 4)\n    dl = torch.utils.data.DataLoader(_ImageSet(paths), batch_size=bs, num_workers=w_eff)\n    n = len(paths)\n    if mmap_path is not None or n > 15000:\n        p = Path(mmap_path) if mmap_path else Path(tempfile.gettempdir()) / f\"pix_mmap_{n}.bin\"\n        mm = np.memmap(str(p), dtype=np.uint8, mode=\"w+\", shape=(n, 3, 224, 224))\n        out = torch.from_numpy(mm)\n    else:\n        out = torch.empty(n, 3, 224, 224, dtype=torch.uint8)\n    k = 0\n    for b in dl:\n        out[k:k + len(b)] = b\n        k += len(b)\n    return out\n\n\ndef tokenize(texts, bs=4096):\n    from transformers import CLIPTokenizer\n    tok = CLIPTokenizer.from_pretrained(NAME)\n    ids_l, att_l = [], []\n    for i in range(0, len(texts), bs):\n        t = tok([str(x) for x in texts[i:i + bs]], padding=\"max_length\", truncation=True, max_length=77, return_tensors=\"pt\")\n        ids_l.append(t[\"input_ids\"].to(torch.int32))\n        att_l.append(t[\"attention_mask\"].to(torch.int8))\n    return torch.cat(ids_l, 0), torch.cat(att_l, 0)\n\n\ndef make_data(frame, *, path_col, text_col, label_col, task_col, task_order, num_classes, workers=8):\n    \"\"\"frame may contain train/val or train/val/test rows (column 'split').\"\"\"\n    assert set(frame[\"split\"]) <= {\"train\", \"val\", \"test\"}, \"unexpected split names\"\n    log(f\"decoding {len(frame)} images ...\")\n    pix = decode_images(frame[path_col].tolist(), workers=workers)\n    ids, att = tokenize(frame[text_col].tolist())\n    splits = frame[\"split\"].to_numpy()\n    return {\"pix\": pix, \"ids\": ids, \"att\": att,\n            \"y\": torch.tensor(frame[label_col].to_numpy(np.int64)),\n            \"train\": splits == \"train\",\n            \"val\": splits == \"val\",\n            \"test\": splits == \"test\",\n            \"task\": frame[task_col].to_numpy(), \"order\": list(task_order), \"C\": int(num_classes)}\n\n\n# ---------------------------------------------------------------------------- model\nclass SessionLoRA(nn.Module):\n    def __init__(self, base, owner):\n        super().__init__()\n        self.base, self.owner = base, owner\n        self.past = nn.ParameterList()\n        self.A = self.B = self.P = None\n\n    def new_session(self):\n        if self.A is not None:\n            A_eff = (self.A.detach() @ self.P) if self.P is not None else self.A.detach()\n            self.past.append(nn.Parameter(A_eff, requires_grad=False))\n            self.past.append(nn.Parameter(self.B.detach(), requires_grad=False))\n        self.P = None if self.owner.proj is None else self.owner.proj.clone()\n        d_in, d_out = self.base.in_features, self.base.out_features\n        dev = self.base.weight.device\n        self.A = nn.Parameter(torch.randn(R, d_in, device=dev) / d_in ** 0.5)\n        self.B = nn.Parameter(torch.zeros(d_out, R, device=dev))\n\n    def forward(self, x):\n        out = self.base(x)\n        if self.A is None:\n            return out\n        for i in range(0, len(self.past), 2):\n            out = out + (x @ self.past[i].T.to(x.dtype) @ self.past[i + 1].T.to(x.dtype)) * (ALPHA / R)\n        A = self.A if self.P is None else self.A @ self.P\n        return out + (x @ A.T.to(x.dtype) @ self.B.T.to(x.dtype)) * (ALPHA / R)\n\n\nclass AttnState:\n    def __init__(self, d, is_text):\n        self.gram = torch.zeros(d, d, dtype=torch.float32, device=DEV)\n        self.proj, self.is_text, self.recording, self.rank = None, is_text, False, 0\n\n    def hook(self, module, inputs):\n        if not self.recording:\n            return\n        with torch.amp.autocast(\"cuda\", enabled=False):\n            x = inputs[0].detach()\n            if self.is_text and TEXT_MASK[\"mask\"] is not None:\n                x = x[TEXT_MASK[\"mask\"].bool()]\n            x = x.reshape(-1, x.shape[-1]).float()\n            self.gram += x.T @ x\n\n    def update_projection(self, tau):\n        if tau is None:\n            return\n        g = torch.nan_to_num(self.gram.double().cpu(), nan=0.0, posinf=0.0, neginf=0.0)\n        g = 0.5 * (g + g.T)\n        evals, evecs = torch.linalg.eigh(g)\n        evals, evecs = evals.flip(0).clamp(min=0), evecs.flip(1)\n        tot = evals.sum().clamp(min=1e-12)\n        k = int((torch.cumsum(evals, 0) / tot < tau).sum().item()) + 1\n        U = evecs[:, :k].float()\n        self.proj = (torch.eye(U.shape[0]) - U @ U.T).to(DEV)\n        self.rank = k\n\n\ndef build(lora=True, towers=(\"vision\", \"text\")):\n    from transformers import CLIPModel\n    m = CLIPModel.from_pretrained(NAME).to(DEV)\n    for p in m.parameters():\n        p.requires_grad_(False)\n    states, loras = [], []\n    if lora:\n        for name, mod in list(m.named_modules()):\n            if isinstance(getattr(mod, \"q_proj\", None), nn.Linear):\n                is_text = name.startswith(\"text_model\")\n                tower_name = \"text\" if is_text else \"vision\"\n                if tower_name not in towers:\n                    continue\n                st = AttnState(mod.q_proj.in_features, is_text)\n                mod.q_proj.register_forward_pre_hook(st.hook)\n                for attr in (\"q_proj\", \"v_proj\"):\n                    lo = SessionLoRA(getattr(mod, attr), st)\n                    setattr(mod, attr, lo)\n                    loras.append(lo)\n                states.append(st)\n    return m.eval(), states, loras\n\n\ndef encode(m, D, idx):\n    pix = ((D[\"pix\"][idx].to(DEV).float() / 255.0) - MEAN.to(DEV)) / STD.to(DEV)\n    ids, att = D[\"ids\"][idx].to(DEV, dtype=torch.long), D[\"att\"][idx].to(DEV, dtype=torch.long)\n    with torch.autocast(\"cuda\", dtype=AMP, enabled=AMP is not None):\n        vo = m.vision_model(pixel_values=pix)\n        i = m.visual_projection(vo[1] if isinstance(vo, tuple) else vo.pooler_output)\n        TEXT_MASK[\"mask\"] = att\n        to = m.text_model(input_ids=ids, attention_mask=att)\n        t = m.text_projection(to[1] if isinstance(to, tuple) else to.pooler_output)\n        TEXT_MASK[\"mask\"] = None\n    return F.normalize(i.float(), dim=-1), F.normalize(t.float(), dim=-1)\n\n\ndef train_session(m, loras, head, D, idx, *, epochs, bs, lr, seed=0):\n    for lo in loras:\n        lo.new_session()\n    params = [p for lo in loras for p in (lo.A, lo.B)] + list(head.parameters())\n    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)\n    use_scaler = (DEV == \"cuda\" and AMP == torch.float16)\n    scaler = (\n        torch.amp.GradScaler(\"cuda\", init_scale=1024.0, enabled=use_scaler)\n        if hasattr(torch, \"amp\")\n        else torch.cuda.amp.GradScaler(init_scale=1024.0, enabled=use_scaler)\n    )\n    g = torch.Generator().manual_seed(seed)\n    m.train()\n    for ep in range(epochs):\n        perm = idx[torch.randperm(len(idx), generator=g)]\n        tot = 0.0\n        for b in range(0, len(perm), bs):\n            s = perm[b:b + bs]\n            i, t = encode(m, D, s)\n            with torch.autocast(\"cuda\", dtype=AMP, enabled=AMP is not None):\n                loss = F.cross_entropy(head(torch.cat([i, t], 1)) * 10, D[\"y\"][s].to(DEV))\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.step(opt)\n            scaler.update()\n            tot += float(loss.detach().item()) * len(s)\n        log(f\"   epoch {ep}: loss {tot / len(perm):.4f}\")\n    m.eval()\n    if DEV == \"cuda\":\n        torch.cuda.empty_cache()\n\n\n@torch.no_grad()\ndef extract(m, D, idx, bs=256):\n    I, T = [], []\n    for b in range(0, len(idx), bs):\n        i, t = encode(m, D, idx[b:b + bs])\n        I.append(i.cpu())\n        T.append(t.cpu())\n    return torch.cat(I), torch.cat(T)\n\n\n# ----------------------------------------------------------------------- evaluation\nPRED = {}          # per-sample validation predictions, exported to predictions.npz for offline analysis\nHEAD_SEEDS = (2026, 2027, 2028)\nBETAS = (0.0, 0.5)\n\n\ndef scores(y_true, y_pred):\n    per_cls = f1_score(y_true, y_pred, average=None, zero_division=0)\n    return {\"macro_f1\": float(np.mean(per_cls)),\n            \"worst_class_f1\": float(np.min(per_cls)),\n            \"acc\": float(accuracy_score(y_true, y_pred))}\n\n\ndef _heads(C, beta):\n    return [BlockAnalyticCL({\"img\": 512, \"txt\": 512}, C, projection_dim=1024, class_balance_beta=beta, seed=s)\n            for s in HEAD_SEEDS]\n\n\ndef _logits(hs, feats, adaptive_sid=False):\n    return sum(h.logits({\"img\": feats[0], \"txt\": feats[1]}, adaptive_sid=adaptive_sid) for h in hs)\n\n\ndef analytic_fit_eval(tr_feats, y_tr, va_feats, y_va, C, key, te_feats=None, y_te=None):\n    out = {}\n    for b in BETAS:\n        hs = _heads(C, b)\n        for h in hs:\n            h.update({\"img\": tr_feats[0], \"txt\": tr_feats[1]}, y_tr)\n        pred = _logits(hs, va_feats).argmax(1).numpy()\n        PRED[f\"{key}/val/beta{b}\"] = pred.astype(np.int16)\n        out[f\"beta_{b}\"] = scores(y_va.numpy(), pred)\n        if te_feats is not None and y_te is not None and len(y_te) > 0:\n            pred_te = _logits(hs, te_feats).argmax(1).numpy()\n            PRED[f\"{key}/test/beta{b}\"] = pred_te.astype(np.int16)\n            out[f\"test_beta_{b}\"] = scores(y_te.numpy(), pred_te)\n        if b == 0.5:\n            for h in hs:\n                h.auto_spectral_ridges()\n            pred_sid = _logits(hs, va_feats, adaptive_sid=True).argmax(1).numpy()\n            out[\"unified_sid\"] = scores(y_va.numpy(), pred_sid)\n            if te_feats is not None and y_te is not None and len(y_te) > 0:\n                pred_sid_te = _logits(hs, te_feats, adaptive_sid=True).argmax(1).numpy()\n                out[\"test_unified_sid\"] = scores(y_te.numpy(), pred_sid_te)\n    return out\n\n\ndef _idx(mask):\n    return torch.tensor(np.where(mask)[0])\n\n\ndef run_g1(D, *, seeds, epochs, bs, lr, stage=\"g1\"):\n    tr, va = _idx(D[\"train\"]), _idx(D[\"val\"])\n    has_te = bool(D.get(\"test\") is not None and np.any(D[\"test\"]))\n    te = _idx(D[\"test\"]) if has_te else None\n    first = _idx(D[\"train\"] & (D[\"task\"] == D[\"order\"][0]))\n    res = {\"frozen\": {}, \"first_session\": {}, \"joint_oracle\": {}}\n    for arm, sub in ((\"frozen\", None), (\"first_session\", first), (\"joint_oracle\", tr)):\n        for seed in ([seeds[0]] if sub is None else seeds):\n            t0 = time.time()\n            torch.manual_seed(seed)\n            m, _, loras = build(lora=sub is not None)\n            if sub is not None:\n                log(f\"[{stage}/{arm}/seed{seed}] LoRA on {len(sub)} samples\")\n                train_session(m, loras, nn.Linear(1024, D[\"C\"]).to(DEV), D, sub, epochs=epochs, bs=bs, lr=lr, seed=seed)\n            f_tr, f_va = extract(m, D, tr), extract(m, D, va)\n            f_te = extract(m, D, te) if has_te else None\n            y_te = D[\"y\"][te] if has_te else None\n            res[arm][f\"seed{seed}\"] = analytic_fit_eval(\n                f_tr, D[\"y\"][tr], f_va, D[\"y\"][va], D[\"C\"], f\"{stage}/{arm}/seed{seed}\", f_te, y_te\n            )\n            res[arm][f\"seed{seed}\"][\"seconds\"] = time.time() - t0\n            log(f\"[{stage}/{arm}/seed{seed}] {res[arm][f'seed{seed}']}\")\n            del m\n            if DEV == \"cuda\":\n                import gc\n                gc.collect()\n                torch.cuda.empty_cache()\n    return res\n\n\ndef run_g2(D, *, taus, seeds, epochs, bs, lr, stage=\"g2\", extra_tower_configs=((\"vision_only_tau0.95\", 0.95, (\"vision\",)),)):\n    va = _idx(D[\"val\"])\n    has_te = bool(D.get(\"test\") is not None and np.any(D[\"test\"]))\n    te = _idx(D[\"test\"]) if has_te else None\n    yv, tv = D[\"y\"][va].numpy(), D[\"task\"][va.numpy()]\n    yte = D[\"y\"][te].numpy() if has_te else None\n    res = {}\n    configs = [((\"naive\" if tau is None else f\"nullspace_tau{tau}\"), tau, (\"vision\", \"text\")) for tau in taus]\n    if extra_tower_configs:\n        configs.extend(extra_tower_configs)\n    for tag, tau, towers in configs:\n        res[tag] = {}\n        for seed in seeds:\n            t0 = time.time()\n            torch.manual_seed(seed)\n            m, states, loras = build(lora=True, towers=towers)\n            head = nn.Linear(1024, D[\"C\"]).to(DEV)\n            heads = {b: _heads(D[\"C\"], b) for b in BETAS}\n            trans_heads = _heads(D[\"C\"], 0.5)\n            stored, ranks, seen = [], [], torch.zeros(D[\"C\"], dtype=torch.bool)\n            acc_matrix = {f\"beta_{b}\": [] for b in BETAS}      # row k: acc on val of task j<=k after session k\n            for k, task in enumerate(D[\"order\"]):\n                idx = _idx(D[\"train\"] & (D[\"task\"] == task))\n                fi_pre, ft_pre = extract(m, D, idx) if k > 0 else (None, None)\n                train_session(m, loras, head, D, idx, epochs=epochs, bs=bs, lr=lr, seed=seed * 100 + k)\n                for st in states:\n                    st.recording = True\n                fi, ft = extract(m, D, idx)\n                for st in states:\n                    st.recording = False\n                    st.update_projection(tau)\n                stored.append((idx, fi, ft))\n                seen[D[\"y\"][idx].unique()] = True\n                for hs in heads.values():\n                    for h in hs:\n                        h.update({\"img\": fi, \"txt\": ft}, D[\"y\"][idx])\n                for h in trans_heads:\n                    if k > 0 and fi_pre is not None:\n                        h.transport_gram({\"img\": fi_pre, \"txt\": ft_pre}, {\"img\": fi, \"txt\": ft})\n                    h.update({\"img\": fi, \"txt\": ft}, D[\"y\"][idx])\n                vi, vt = extract(m, D, va)\n                for b, hs in heads.items():\n                    lg = _logits(hs, (vi, vt))\n                    lg[:, ~seen] = -float(\"inf\")\n                    pred = lg.argmax(1).numpy()\n                    acc_matrix[f\"beta_{b}\"].append([float((pred[tv == t] == yv[tv == t]).mean()) for t in D[\"order\"][:k + 1]])\n                ranks.append(float(np.mean([st.rank / st.gram.shape[0] for st in states])) if tau else 0.0)\n                log(f\"[{stage}/{tag}/seed{seed}] session {k + 1}/{len(D['order'])} n={len(idx)} \"\n                    f\"protected-rank {ranks[-1]:.3f} acc(b=0) {np.mean(acc_matrix['beta_0.0'][-1]):.4f} ({time.time() - t0:.0f}s)\")\n            r = {\"stale\": {}, \"transport_sid\": {}, \"acc_matrix\": acc_matrix, \"protected_rank_frac\": ranks, \"towers\": list(towers)}\n            ti, tt = extract(m, D, te) if has_te else (None, None)\n            for b, hs in heads.items():\n                pred = _logits(hs, (vi, vt)).argmax(1).numpy()\n                PRED[f\"{stage}/{tag}/seed{seed}/stale/val/beta{b}\"] = pred.astype(np.int16)\n                r[\"stale\"][f\"beta_{b}\"] = scores(yv, pred)\n                if has_te:\n                    pred_te = _logits(hs, (ti, tt)).argmax(1).numpy()\n                    PRED[f\"{stage}/{tag}/seed{seed}/stale/test/beta{b}\"] = pred_te.astype(np.int16)\n                    r[\"stale\"][f\"test_beta_{b}\"] = scores(yte, pred_te)\n            for h in trans_heads:\n                h.auto_spectral_ridges()\n            pred_tr_val = _logits(trans_heads, (vi, vt), adaptive_sid=True).argmax(1).numpy()\n            r[\"transport_sid\"][\"val\"] = scores(yv, pred_tr_val)\n            if has_te:\n                pred_tr_te = _logits(trans_heads, (ti, tt), adaptive_sid=True).argmax(1).numpy()\n                r[\"transport_sid\"][\"test\"] = scores(yte, pred_tr_te)\n            all_idx = torch.cat([s[0] for s in stored])\n            fi, ft = extract(m, D, all_idx)\n            si, st_ = torch.cat([s[1] for s in stored]), torch.cat([s[2] for s in stored])\n            di, dt = (fi - si).norm(dim=1), (ft - st_).norm(dim=1)\n            r[\"drift_img\"], r[\"drift_txt\"] = float(di.mean()), float(dt.mean())\n            bounds = np.cumsum([0] + [len(s[0]) for s in stored])\n            r[\"drift_by_session\"] = [{\"img\": float(di[a:b].mean()), \"txt\": float(dt[a:b].mean())}\n                                     for a, b in zip(bounds[:-1], bounds[1:])]\n            r[\"refit\"] = analytic_fit_eval((fi, ft), D[\"y\"][all_idx], (vi, vt), D[\"y\"][va], D[\"C\"],\n                                           f\"{stage}/{tag}/seed{seed}/refit\", (ti, tt) if has_te else None, D[\"y\"][te] if has_te else None)\n            r[\"seconds\"] = time.time() - t0\n            log(f\"[{stage}/{tag}/seed{seed}] stale {r['stale']} | refit {r['refit']} | \"\n                f\"drift img {r['drift_img']:.4f} txt {r['drift_txt']:.4f}\")\n            res[tag][f\"seed{seed}\"] = r\n            del m\n            if DEV == \"cuda\":\n                import gc\n                gc.collect()\n                torch.cuda.empty_cache()\n    return res\n\n\ndef _mean_over_seeds(per_seed, metric, field=None, prefix=\"beta\"):\n    \"\"\"Best beta (by seed-mean) of the seed-mean metric.\"\"\"\n    rows = [v[field] if field else v for v in per_seed.values()]\n    return max(float(np.mean([r[f\"{prefix}_{b}\"][metric] for r in rows if f\"{prefix}_{b}\" in r])) for b in BETAS)\n\n\ndef verdicts(g1, g2, metric):\n    fz, orc = _mean_over_seeds(g1[\"frozen\"], metric), _mean_over_seeds(g1[\"joint_oracle\"], metric)\n    out = {\"metric\": metric, \"frozen\": fz, \"joint_oracle\": orc,\n           \"first_session\": _mean_over_seeds(g1[\"first_session\"], metric),\n           \"G1_oracle_gain_pts\": 100 * (orc - fz), \"G1\": \"PASS\" if 100 * (orc - fz) >= 2 else \"FAIL\"}\n    if g2:\n        target = fz + 0.5 * (orc - fz)\n        stale = {k: _mean_over_seeds(v, metric, \"stale\") for k, v in g2.items()}\n        trans_sid = {k: float(np.mean([s[\"transport_sid\"][\"val\"][metric] for s in v.values()]))\n                     for k, v in g2.items() if \"transport_sid\" in next(iter(v.values()))}\n        cand = [v for k, v in stale.items() if k != \"naive\"] + [v for k, v in trans_sid.items() if k != \"naive\"]\n        out.update({\"G2_target\": target, \"G2_stale\": stale, \"G2_transport_sid\": trans_sid,\n                    \"G2\": \"PASS\" if max(cand) >= target else \"FAIL\"})\n        first_seed = next(iter(next(iter(g2.values())).values()))\n        if \"test_beta_0.5\" in first_seed.get(\"stale\", {}):\n            out[\"G3_test_frozen\"] = _mean_over_seeds(g1[\"frozen\"], metric, prefix=\"test_beta\")\n            out[\"G3_test_first_session\"] = _mean_over_seeds(g1[\"first_session\"], metric, prefix=\"test_beta\")\n            out[\"G3_test_joint_oracle\"] = _mean_over_seeds(g1[\"joint_oracle\"], metric, prefix=\"test_beta\")\n            out[\"G3_test_stale\"] = {k: _mean_over_seeds(v, metric, \"stale\", prefix=\"test_beta\") for k, v in g2.items()}\n            out[\"G3_test_transport_sid\"] = {\n                k: float(np.mean([s[\"transport_sid\"][\"test\"][metric] for s in v.values()]))\n                for k, v in g2.items() if \"test\" in next(iter(v.values())).get(\"transport_sid\", {})\n            }\n    return out\n\n\ndef export(out_dir, name, D, payload):\n    \"\"\"results JSON + per-sample validation/test predictions (for paired bootstrap offline).\"\"\"\n    import json\n    import platform\n    out_dir.mkdir(parents=True, exist_ok=True)\n    va = np.where(D[\"val\"])[0]\n    te = np.where(D[\"test\"])[0] if D.get(\"test\") is not None else np.array([], dtype=int)\n    meta = {\"torch\": torch.__version__, \"python\": platform.python_version(), \"device\": DEV,\n            \"gpu\": torch.cuda.get_device_name(0) if DEV == \"cuda\" else \"cpu\", \"amp\": str(AMP),\n            \"n_train\": int(D[\"train\"].sum()), \"n_val\": int(len(va)), \"n_test\": int(len(te)), \"num_classes\": D[\"C\"],\n            \"task_order\": [str(t) for t in D[\"order\"]], \"head_seeds\": list(HEAD_SEEDS), \"betas\": list(BETAS),\n            \"lora\": {\"r\": R, \"alpha\": ALPHA, \"targets\": [\"q_proj\", \"v_proj\"], \"towers\": [\"vision\", \"text\"]}}\n    (out_dir / f\"{name}_results.json\").write_text(json.dumps({\"meta\": meta, **payload}, indent=2))\n    keep = {k.replace(\"/\", \"__\"): v for k, v in PRED.items() if k.split(\"/\")[0] in payload[\"stages\"]}\n    extra = {}\n    if len(te) > 0:\n        extra[\"y_test\"] = D[\"y\"][te].numpy().astype(np.int16)\n        extra[\"task_test\"] = np.array([str(t) for t in D[\"task\"][te]])\n    np.savez_compressed(out_dir / f\"{name}_predictions.npz\", y_val=D[\"y\"][va].numpy().astype(np.int16),\n                        task_val=np.array([str(t) for t in D[\"task\"][va]]), **extra, **keep)\n"}
for p, s in FILES.items():
    q = pathlib.Path('/content/h1code') / p
    q.parent.mkdir(parents=True, exist_ok=True)
    q.write_text(s)
import sys; sys.path.insert(0, '/content/h1code')
import h1_gate_engine as E
print('engine on', E.DEV, '| amp', E.AMP)


In [ ]:
# ---- experiment configuration ----
EPOCHS, BS, LR = 2, 64, 1e-4
TAUS = [None, 0.95, 0.99]           # None = naive sequential LoRA
SEEDS_FOOD = [2026]                 # add 2027, 2028 if time allows
WORKERS = min(os.cpu_count() or 2, 4)
import json, pathlib, numpy as np, torch
OUT = pathlib.Path('/content/h1_results'); OUT.mkdir(exist_ok=True)
E.LOG_PATH = str(OUT / 'run.log')
CONFIG = dict(epochs=EPOCHS, bs=BS, lr=LR, taus=TAUS, seeds_food=SEEDS_FOOD,
              g1_rule='oracle - frozen >= 2 pts', g2_rule='best null-space stale/transport >= frozen + 0.5*(oracle - frozen)')
(OUT / 'config.json').write_text(json.dumps(CONFIG, indent=2))
from google.colab import files
def download():
    !cd /content && rm -f h1_results.zip && zip -q -r h1_results.zip h1_results
    try: files.download('/content/h1_results.zip')
    except Exception as e: print('download failed (results still in /content/h1_results):', e)


## 1. Download & Prepare UPMC Food-101 (RebQ/PAL Protocol)


In [ ]:
# Zero-config multi-connection aria2c download from Kaggle (16 streams, no KAGGLE_USERNAME required)
if not pathlib.Path('/content/food101').exists() or not list(pathlib.Path('/content/food101').rglob('*.csv')):
    !apt-get -qq install -y aria2 > /dev/null
    pathlib.Path('/content/food101').mkdir(parents=True, exist_ok=True)
    !aria2c -x 16 -s 16 -k 1M --allow-overwrite=true -q "https://www.kaggle.com/api/v1/datasets/download/gianmarco96/upmcfood101" -d /content -o food101.zip
    !unzip -q -o /content/food101.zip -d /content/food101 && rm -f /content/food101.zip
!find /content/food101 -maxdepth 3 -type d | head -20; find /content/food101 -name '*.csv'


In [ ]:
import gc, json, pathlib, glob, numpy as np, pandas as pd, torch
root = pathlib.Path('/content/food101')
csv = [p for p in glob.glob(str(root / '**/*.csv'), recursive=True) if 'train' in pathlib.Path(p).name.lower()][0]
df = pd.read_csv(csv, header=None, names=['image', 'text', 'label'], on_bad_lines='skip')
if str(df.iloc[0]['label']).lower() in {'label', 'category', 'class'}: df = df.iloc[1:].reset_index(drop=True)
index = {pathlib.Path(p).name: p for p in glob.glob(str(root / '**/train/**/*.jpg'), recursive=True)}
df['path'] = df['image'].map(lambda n: index.get(pathlib.Path(str(n)).name))
print('raw train rows', len(df), '| missing images', int(df['path'].isna().sum()), '| csv', csv)
df = df.dropna(subset=['path']).reset_index(drop=True)
df = df[df['label'] != df['label'].value_counts().idxmin()].reset_index(drop=True)   # RebQ: drop rarest class
classes = sorted(df['label'].unique()); assert len(classes) == 100, len(classes)
df['y'] = df['label'].map({c: i for i, c in enumerate(classes)})
PER_CLASS = 120   # T4 12.7GB RAM-safe stratified subset: 120/class x 100 classes = 12,000 samples (1.8 GB tensor)
rng = np.random.RandomState(0); keep_rows = []
for c, idx in df.groupby('y').groups.items():
    idx = rng.permutation(np.array(idx))[:PER_CLASS]
    keep_rows.append(idx)
df = df.loc[np.concatenate(keep_rows)].reset_index(drop=True)
df['split'] = 'train'
for c, idx in df.groupby('y').groups.items():                                         # 80% train, 10% val, 10% test
    idx = np.array(idx); n_va = max(1, len(idx) // 10)
    df.loc[idx[:n_va], 'split'] = 'val'
    df.loc[idx[n_va:2 * n_va], 'split'] = 'test'
corder = np.random.RandomState(2026).permutation(100)
df['task'] = df['y'].map({int(c): k // 10 for k, c in enumerate(corder)})
del index; gc.collect()
print(df['split'].value_counts().to_dict(), '| total rows:', len(df))
D_food = E.make_data(df, path_col='path', text_col='text', label_col='y', task_col='task',
                     task_order=list(range(10)), num_classes=100, workers=WORKERS)


## 2. Run Gate G1 & Gate G2 on UPMC Food-101


In [ ]:
# G1: frozen / first-session LoRA / joint-oracle LoRA
import json, pathlib, numpy as np, torch
counts_f = json.loads(df.groupby(['task', 'split']).size().unstack(fill_value=0).to_json())
g1f = E.run_g1(D_food, seeds=SEEDS_FOOD, epochs=EPOCHS, bs=BS, lr=LR, stage='food_g1')
vf = E.verdicts(g1f, None, 'acc'); print(json.dumps(vf, indent=2))
E.export(OUT / 'food101', 'food', D_food, {'stages': ['food_g1'], 'config': CONFIG, 'counts': counts_f,
                                          'g1': g1f, 'verdicts': vf})
download()


In [ ]:
# G2: 10 class-incremental tasks (naive vs null-space LoRA + BlockAnalyticCL & Transport-SID)
import json, pathlib, numpy as np, torch
g2f = E.run_g2(D_food, taus=TAUS, seeds=SEEDS_FOOD, epochs=EPOCHS, bs=BS, lr=LR, stage='food_g2')
vf = E.verdicts(g1f, g2f, 'acc'); print(json.dumps(vf, indent=2))
E.export(OUT / 'food101', 'food', D_food, {'stages': ['food_g1', 'food_g2'], 'config': CONFIG, 'counts': counts_f,
                                          'g1': g1f, 'g2': g2f, 'verdicts': vf})
download()


## 3. Summary


In [ ]:
for p in sorted(OUT.glob('*/*_results.json')):
    print(p.name, json.dumps(json.loads(p.read_text())['verdicts'], indent=2))
!ls -la /content/h1_results/*
download()
